In [1]:
import pandas as pd 
import numpy as np 

# 1. Klienti un pamata parāda pazīmes

In [2]:
# Automātiski atjaunojams fails ar datiem par visiem klientiem 
clients_path = r"G:\GSCLV-FIN PUBLIC\-=DATI=-\Report Automatization TheTable\The Table.csv"
clients_df = pd.read_csv(
    clients_path,
    sep=';',
    engine='python',
    on_bad_lines='warn',
    encoding='utf-8',
    encoding_errors='replace'
)

# Atstāt noteiktas kolonnas clients_df:
clients_clean = clients_df[['File', 
                            'Client number', # client ID
                            'Legal start', # for legal_flag
                            'Principal', # debt amount
                            'Charged', # purchase date
                            'LastPaym.', # last payment date
                            'Home', # is_phone flag
                            'Cell',
                            'Email', # is_email flag
                            'ZIP', # region
                            'Type', # private or business
                            'Interest', # interest rate
                            'Accrued', # accrued interest
                            'Fees', # fees
                            'Legal fees', # legal fees
                            'Miscellaneous', 
                            'Other charges',
                            'Agree.Date', # agreement date
                            'Termin.date'
                            ]]


In [3]:
# Pārbaudīt, cik daudz vērtību sakrīt  kolonnās 'Agree.Date' un 'Termin.date':
clients_clean['Agree.Date'] = pd.to_datetime(clients_clean['Agree.Date'], errors='coerce', format='%d/%m/%Y')
clients_clean['Termin.date'] = pd.to_datetime(clients_clean['Termin.date'], errors='coerce', format='%d/%m/%Y')
equal_mask = clients_clean["Agree.Date"] == clients_clean["Termin.date"]
equal_count = equal_mask.sum()
total_count = len(clients_clean)

print("Equal values:", equal_count)
print("Percentage:", round(equal_count / total_count * 100, 2), "%")


Equal values: 58888
Percentage: 96.5 %


In [4]:
clients_to_remove = [200, 2001, 2002, 20000, 20028, 20045, 20110, 20136, 20167, 20168, 20191, 20207, 20222, 20253, 20279]
clients_clean['Charged'] = pd.to_datetime(clients_clean['Charged'], errors='coerce', format='%d/%m/%Y')
clients_clean = clients_clean[
    (clients_clean['Charged'] <= '2025-12-31') & (clients_clean['Type'] == 'PI') &
    (clients_clean['Client number'].isin(clients_to_remove) == False)]

# 2. Vecums un dzimums

In [5]:
# R ģenerēts fails ar klientu dzimumu un vecumu:
age_gender_path = r"G:\GSCLV-FIN\Current month BI Reports\Client profile\2026\Valuation data\clients_age_gender.csv"
age_gender_df = pd.read_csv(age_gender_path, sep=';')

age_gender_df['age_group'].value_counts(dropna=False)

# Aizvietot NaN vērtības ar "Unknown"
age_gender_df['age_group'] = age_gender_df['age_group'].fillna('Unknown')
age_gender_df['gender'] = age_gender_df['gender'].fillna('Unknown')

# 3. Kavējuma datums

In [6]:
# kavējuma datumu fails 
delay_path = r"G:\GSCLV-FIN\Current month BI Reports\Client profile\2026\Valuation data\Kavejuma_datums.xlsx"
delay_df = pd.read_excel(delay_path)

# Noņemam nevajadzīgo kolonnu 'Portfeļa nosaukums' no delay_df
delay_df = delay_df.drop(columns=['Portfeļa nosaukums'])

# 4. Reģioni

In [7]:
regions_path = r"G:\GSCLV-FIN\Current month BI Reports\Client profile\2026\Valuation data\regions.csv"
regions_df = pd.read_csv(
    regions_path,
    sep=';',
    engine='python',
    encoding='cp1257',
    on_bad_lines='warn'
)

# Pārdēvēt Pasta Indekss kolonnu nosaukumu uz "ZIP", lai skaidri norādītu, ka tā attiecas uz pasta indeksu unificēšanai:
regions_df = regions_df.rename(columns={'Pasta Indekss': 'ZIP'})

regions_df["region_len"] = regions_df["Reģions"].str.len()

regions_df_sorted = regions_df.sort_values(["ZIP", "region_len"])
regions_df_dedup = regions_df_sorted.drop_duplicates(subset="ZIP", keep="first").drop(columns="region_len")

# Pārbaudām, vai nav ZIP dublikātu:
duplicate_zips = regions_df_dedup['ZIP'].duplicated().sum()
print(f"Number of duplicate 'ZIP' entries: {duplicate_zips}")

Number of duplicate 'ZIP' entries: 0


# 5. Datu Apvienošana

In [8]:
# Apvienojam datus pa 'File' kolonnu, lai iegūtu vienu kopīgu datu kopu:
combined_df = pd.merge(clients_clean, age_gender_df, on='File', how='left')
combined_df2 = pd.merge(combined_df, delay_df, on='File', how='left')
combined_df3 = pd.merge(combined_df2, regions_df_dedup, on='ZIP', how='left')


In [9]:
# Izdzēšam nevajadzīgos failus, lai atbrīvotu atmiņu:
del clients_df, clients_clean, age_gender_df, delay_df, regions_df_dedup, combined_df, combined_df2

Izveidojam combined_cl_df: 
1. izveidojam legal_flag 
2. contact_flag
3. mainām uz datetime formātu kolonnas: Closed, Agree.Date, Termin.date

In [10]:
combined_cl_df = combined_df3.copy()

combined_cl_df['legal_flag'] = combined_cl_df['Legal start'].notnull().astype(int)
combined_cl_df['contact_flag'] = ((combined_cl_df['Home'].notnull()) | (combined_cl_df['Cell'].notnull()) | (combined_cl_df['Email'].notnull())).astype(int)

combined_cl_df['Agree.Date'] = pd.to_datetime(combined_cl_df['Agree.Date'], errors='coerce', format='%d/%m/%Y')
combined_cl_df['LastPaym.'] = pd.to_datetime(combined_cl_df['LastPaym.'], errors='coerce', format='%d/%m/%Y')

columns_to_drop = ['Legal start', 'Home', 'Cell', 'Email', 'ZIP', 'Type', 'Client_nr']
combined_cl_df = combined_cl_df.drop(columns=columns_to_drop)

In [11]:
# Nolasīt Excel failu ar User 3 datiem:
valuation_df = pd.read_excel(r"G:\GSCLV-FIN PUBLIC\-=DATI=-\Valuation dates\Valuation 2026-07.xlsx")

In [12]:
# Atstāt tikai kolonnas 'Client number', 'File', 'User 3 Pakalpojums ', 'Detail 4 Valuation model'
# ['Client number', 'File', 'User 3 Pakalpojums ', 'Score initial Valuation ', 'Detail 2 Oriģinālais kavējuma datums ', 'Detail 4 Valuation model', 'UGF']
valuation_cl_df = valuation_df[['Client number', 'File', 'User 3 Pakalpojums ', 'Detail 4 Valuation model']]

In [13]:
# Apvienojam datus pa 'Client number' un 'File' kolonnu, lai iegūtu vienu kopīgu datu kopu:
combined_valuation_df = pd.merge(combined_cl_df, valuation_cl_df, on=['Client number', 'File'], how='left')
# Rindu skaits
combined_valuation_df.shape[0]

55775

In [14]:
# Atstājam tikai 'Detail 4 Valuation model' = 'Pre-court' un dzēšam šo kolonnu
precourt_df = combined_valuation_df[combined_valuation_df['Detail 4 Valuation model'] == 'Pre-court'].drop(columns=['Detail 4 Valuation model'])

# Pārdēvēt kolonnu 'User 3 Pakalpojums ' uz 'debt_type' un aizvietot NaN vērtības ar "Unknown"
precourt_df = precourt_df.rename(columns={'User 3 Pakalpojums ': 'debt_type'})
precourt_df['debt_type'] = precourt_df['debt_type'].fillna('Unknown')

precourt_df.shape[0]

53365

In [15]:
del combined_cl_df, combined_valuation_df, valuation_cl_df, valuation_df

In [16]:
# Izveidot kolonnu interest_wo_accrued, kas aprēķina procentu summu bez uzkrātajiem procentiem:
precourt_df['interest_wo_accrued'] = precourt_df['Interest'] - precourt_df['Accrued']

# Dzēst nevajadzīgās kolonnas, kas vairs nav nepieciešamas:
columns_to_drop = ['Interest', 'Accrued']
precourt_df = precourt_df.drop(columns=columns_to_drop)



In [17]:
# Rindu skaits, kur interest_wo_accrued ir mazāks par 0:
precourt_df[precourt_df['interest_wo_accrued'] < 0].shape[0]

17

In [18]:
# Pārdēvēt kolonnu Kavējuma datums uz 'Delay date' un aizvietot NaN vērtības un '1900-01-01 00:00:00' vērtības ar "Unknown":
precourt_df.rename(columns={'Kavējuma datums': 'Delay date'}, inplace=True)



In [19]:
# Replace 1900-01-01 with missing datetime
precourt_df.loc[
    precourt_df["Delay date"] == pd.Timestamp("1900-01-01 00:00:00"),
    "Delay date"
] = pd.NaT

# Replace missing datetime with NaT
precourt_df['Delay date'] = pd.to_datetime(precourt_df['Delay date'], errors='coerce', format='%d/%m/%Y')
precourt_df['Delay date'] = precourt_df['Delay date'].fillna(pd.NaT)

In [22]:
# Daram to pašu ar LastPaym.
precourt_df.loc[
    precourt_df["LastPaym."] == pd.Timestamp("1900-01-01 00:00:00"),
    "LastPaym."
] = pd.NaT

# Replace missing datetime with NaT
precourt_df['LastPaym.'] = pd.to_datetime(precourt_df['LastPaym.'], errors='coerce', format='%d/%m/%Y')
precourt_df['LastPaym.'] = precourt_df['LastPaym.'].fillna(pd.NaT)

In [23]:
final_df = precourt_df.copy()

# Izveidojam jaunas kolonnas

final_df["days_lastpaym_to_charged"] = (
    final_df["Charged"] - final_df["LastPaym."]
).dt.days

final_df["days_delay_to_lastpaym"] = (
    final_df["LastPaym."] - final_df["Delay date"]
).dt.days

final_df["days_agreement_to_delay"] = (
    final_df["Delay date"] - final_df["Agree.Date"]
).dt.days

# Pārbaudīt, vai LastPaym. ir pēc Kavējuma datums un izveidot jaunu kolonnu "paid_after_first_default"
final_df["paid_after_first_default"] = (
    final_df["LastPaym."].notna()
    & final_df["Delay date"].notna()
    & (final_df["LastPaym."] >= final_df["Delay date"])
).astype("int8")


In [ ]:
# Atlasīt un izdrukāt rindas, kur LastPaym > Charged, kas nozīmē, ka pēdējais maksājums ir veikts pēc cesijas datuma:
precourt_df[precourt_df["LastPaym."] > precourt_df["Charged"]]


Maksājumi
Ja Sales ir NA = 0
Ja ir kods: 
Nauda kura paliek uz operatoru (F)

0 – izfiltrējam, 1 atstājam
"G:\GSCLV-FIN PUBLIC\-=DATI=-\Transaction codes un Case status\Case status.xlsx"
